# Tokenization from Scratch

## Byte-Level BPE, GPT-2-Style Tokenization, and SentencePiece

This notebook is a hands-on study of how text is converted into token IDs for language models.

The implementation starts from raw Unicode text, converts it to UTF-8 bytes, learns Byte Pair Encoding (BPE) merge rules, builds a vocabulary, and then implements encoding and decoding. After the from-scratch implementation, the notebook connects those ideas to GPT-2-style pre-tokenization, `tiktoken`, GPT-2 tokenizer files, and SentencePiece.

### Learning objectives

By the end of the notebook, you should understand:

- Why text is converted to bytes before byte-level BPE.
- Why there are 256 possible base byte values.
- Why learned BPE token IDs can start at `256`.
- How pair frequencies are counted.
- How BPE merge rules are learned.
- How a learned vocabulary is reconstructed.
- How encoding applies merge rules in rank order.
- How decoding reconstructs the original UTF-8 text.
- Where GPT-2-style regex splitting fits into the pipeline.
- What `encoder.json`, `vocab.bpe`, and special tokens represent.
- How SentencePiece can train a BPE tokenizer from raw text.

> **Important:** This is an educational implementation. It is intentionally written for clarity rather than production speed or memory efficiency.


## 1. Setup

The notebook uses a small set of Python packages for the later sections.

For a fresh environment, install them first. The repository also contains `requirements.txt` so the same environment can be recreated outside the notebook.

In [1]:
%pip install -q regex tiktoken sentencepiece

from pathlib import Path
from urllib.request import urlretrieve
import json
import os

print("Setup complete.")

Note: you may need to restart the kernel to use updated packages.
Setup complete.


## 2. The Big Picture

A simplified byte-level BPE pipeline looks like this:

```text
Raw Unicode text
       ↓
UTF-8 encoding
       ↓
Byte IDs: 0–255
       ↓
Count adjacent pairs
       ↓
Merge the most frequent pair
       ↓
Create a new token ID: 256+
       ↓
Repeat many times
       ↓
Final token sequence
```

The key distinction is:

```text
0–255  → the possible byte values
256+   → learned tokens representing sequences of bytes
```

So token `256` is **not** a new byte. It is a learned vocabulary entry representing a byte sequence produced by a merge.


## 3. Unicode Text → UTF-8 Bytes

Python strings contain Unicode text. Byte-level tokenization first converts that text into UTF-8.

ASCII characters normally use one byte, while many Unicode characters require multiple bytes. This allows the tokenizer to represent arbitrary Unicode text without creating one base vocabulary entry for every Unicode code point.

In [2]:
text = (
    "Unicode! ⓊⓃⒾⒸⓄⒹⒺ! 😁 "
    "The very name strikes fear and awe into the hearts of programmers worldwide. "
    "We all know we ought to “support Unicode” in our software."
)

tokens = list(text.encode("utf-8"))

print("Original text:")
print(text)

print("\nFirst 60 UTF-8 byte values:")
print(tokens[:60])

print("\nCharacter count:", len(text))
print("UTF-8 byte count:", len(tokens))

Original text:
Unicode! ⓊⓃⒾⒸⓄⒹⒺ! 😁 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software.

First 60 UTF-8 byte values:
[85, 110, 105, 99, 111, 100, 101, 33, 32, 226, 147, 138, 226, 147, 131, 226, 146, 190, 226, 146, 184, 226, 147, 132, 226, 146, 185, 226, 146, 186, 33, 32, 240, 159, 152, 129, 32, 84, 104, 101, 32, 118, 101, 114, 121, 32, 110, 97, 109, 101, 32, 115, 116, 114, 105, 107, 101, 115, 32, 102]

Character count: 155
UTF-8 byte count: 176


### Why 256 base tokens?

A byte is an integer from `0` through `255`, giving exactly:

\[
2^8 = 256
\]

possible byte values.

A byte-level tokenizer can therefore use these 256 values as its initial vocabulary. It then grows the vocabulary by learning new merged symbols.

In [3]:
print("Number of possible byte values:", 256)
print("Minimum byte value:", 0)
print("Maximum byte value:", 255)

assert set(tokens).issubset(set(range(256)))
print("All encoded values are valid byte IDs.")

Number of possible byte values: 256
Minimum byte value: 0
Maximum byte value: 255
All encoded values are valid byte IDs.


## 4. Counting Consecutive Pairs

BPE repeatedly looks at adjacent token pairs.

For:

```text
[97, 98, 97, 98, 99]
```

the consecutive pairs are:

```text
(97, 98)
(98, 97)
(97, 98)
(98, 99)
```

Therefore `(97, 98)` occurs twice and can be selected for the next merge.

In [4]:
def get_stats(ids):
    # Count the frequency of every adjacent pair.
    counts = {}

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1

    return counts


example = [97, 98, 97, 98, 99]
stats = get_stats(example)

print("Pair frequencies:")
for pair, count in stats.items():
    print(f"{pair}: {count}")

top_pair = max(stats, key=stats.get)
print("\nMost frequent pair:", top_pair)
print("Frequency:", stats[top_pair])

Pair frequencies:
(97, 98): 2
(98, 97): 1
(98, 99): 1

Most frequent pair: (97, 98)
Frequency: 2


## 5. Merging a Pair

Once BPE chooses a pair, every **non-overlapping** occurrence is replaced by a new token ID.

Example:

```text
Before:
[97, 98, 97, 98, 99]

Merge:
(97, 98) → 256

After:
[256, 256, 99]
```

The new ID `256` represents the concatenation of the two original byte symbols.

In [5]:
def merge(ids, pair, new_id):
    # Replace every non-overlapping occurrence of pair with new_id.
    new_ids = []
    i = 0

    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1

    return new_ids


merged = merge(example, (97, 98), 256)

print("Before:", example)
print("After: ", merged)

assert merged == [256, 256, 99]
print("Merge check passed.")

Before: [97, 98, 97, 98, 99]
After:  [256, 256, 99]
Merge check passed.


## 6. Train a Tiny Byte-Level BPE Model

Now we repeat the process.

For every merge step:

1. Count all adjacent pairs.
2. Select the most frequent pair.
3. Assign the next vocabulary ID.
4. Merge the pair.
5. Store the rule.

If the base vocabulary contains 256 byte symbols and we want a vocabulary of 276 entries:

```text
276 - 256 = 20
```

so we learn 20 merge rules.

A tiny vocabulary is used here only so that the learned rules remain easy to inspect.

In [6]:
training_text = (
    "In this BPE tokenization process, we start with a piece of text and convert it "
    "into a sequence of bytes. We then find consecutive pairs of tokens and count "
    "how frequently each pair appears. The most frequent pair is merged into a new "
    "token, and this process is repeated multiple times so that the tokenizer "
    "gradually learns larger pieces of text."
)

training_ids = list(training_text.encode("utf-8"))

vocab_size = 276
num_merges = vocab_size - 256

ids = list(training_ids)
merges = {}

for i in range(num_merges):
    stats = get_stats(ids)

    if not stats:
        break

    pair = max(stats, key=stats.get)
    new_id = 256 + i

    ids = merge(ids, pair, new_id)
    merges[pair] = new_id

print("Base vocabulary size:", 256)
print("Requested vocabulary size:", vocab_size)
print("Learned merge rules:", len(merges))

print("\nLearned merges:")
for pair, new_id in merges.items():
    print(f"{pair} -> {new_id}")

Base vocabulary size: 256
Requested vocabulary size: 276
Learned merge rules: 20

Learned merges:
(32, 116) -> 256
(115, 32) -> 257
(101, 110) -> 258
(116, 32) -> 259
(256, 104) -> 260
(32, 112) -> 261
(99, 101) -> 262
(32, 97) -> 263
(100, 32) -> 264
(105, 257) -> 265
(256, 111) -> 266
(266, 107) -> 267
(267, 258) -> 268
(97, 114) -> 269
(111, 102) -> 270
(101, 114) -> 271
(97, 116) -> 272
(111, 110) -> 273
(110, 264) -> 274
(274, 99) -> 275


## 7. Understanding the Merge Table

The merge dictionary maps:

```text
(old_token_1, old_token_2) → new_token_id
```

An important detail is that old tokens are allowed to be either:

- one of the original byte IDs `0–255`, or
- an already learned token such as `256`, `257`, etc.

That is how BPE gradually builds larger structures from smaller ones.

In [7]:
print("Number of merge rules:", len(merges))

for rank, ((p0, p1), new_id) in enumerate(merges.items(), start=1):
    print(f"Merge #{rank:02d}: ({p0}, {p1}) -> {new_id}")

    if rank == 5:
        break

Number of merge rules: 20
Merge #01: (32, 116) -> 256
Merge #02: (115, 32) -> 257
Merge #03: (101, 110) -> 258
Merge #04: (116, 32) -> 259
Merge #05: (256, 104) -> 260


## 8. Reconstructing the Vocabulary

For the base vocabulary:

```python
vocab[id] = bytes([id])
```

For a learned token, its byte representation is formed by concatenating the byte representations of the two tokens that created it.

For example:

```text
(101, 32) → 256
```

means:

```text
vocab[256] = vocab[101] + vocab[32]
```

This gives us a complete mapping from token IDs back to byte sequences.

In [8]:
vocab = {idx: bytes([idx]) for idx in range(256)}

for (p0, p1), new_id in merges.items():
    vocab[new_id] = vocab[p0] + vocab[p1]

print("Vocabulary entries:", len(vocab))
print("Byte token 65:", vocab[65])
print("Learned token example:", next(iter(merges.values())))

assert all(idx in vocab for idx in range(256))
assert len(vocab) == 256 + len(merges)
print("Vocabulary construction check passed.")

Vocabulary entries: 276
Byte token 65: b'A'
Learned token example: 256
Vocabulary construction check passed.


## 9. Encoding with the Learned BPE Rules

During encoding, text is first converted to UTF-8 bytes.

Then, while at least one known merge can be applied:

- inspect the current adjacent pairs,
- find the pair with the smallest learned merge rank,
- apply that merge,
- repeat.

Why the smallest rank?

Because multiple known pairs may exist at the same time. The learned merge order determines which one must be applied first. This makes encoding deterministic.

In [9]:
def encode(text):
    ids = list(text.encode("utf-8"))

    while len(ids) >= 2:
        stats = get_stats(ids)

        pair = min(
            stats,
            key=lambda p: merges.get(p, float("inf"))
        )

        if pair not in merges:
            break

        ids = merge(ids, pair, merges[pair])

    return ids


sample = "In this BPE tokenizer"
encoded = encode(sample)

print("Input:")
print(sample)

print("\nEncoded token IDs:")
print(encoded)

print("\nToken count:", len(encoded))
print("UTF-8 byte count:", len(sample.encode("utf-8")))

Input:
In this BPE tokenizer

Encoded token IDs:
[73, 110, 260, 265, 66, 80, 69, 268, 105, 122, 271]

Token count: 11
UTF-8 byte count: 21


## 10. Decoding

Decoding performs the inverse representation step:

```text
token IDs
   ↓
byte sequences from vocabulary
   ↓
concatenate bytes
   ↓
UTF-8 decode
   ↓
original text
```

The decoder does not need to know the original merge history. The final vocabulary already stores the byte sequence represented by each token.

In [10]:
def decode(ids):
    token_bytes = [vocab[idx] for idx in ids]
    return b"".join(token_bytes).decode("utf-8")


decoded = decode(encoded)

print("Decoded:")
print(decoded)

assert decoded == sample
print("\nRound-trip check passed.")

Decoded:
In this BPE tokenizer

Round-trip check passed.


## 11. Compression Effect of BPE

For this toy corpus, BPE may reduce the number of symbols required to represent repeated patterns.

This does **not** mean that BPE always produces fewer tokens for every piece of text. The result depends on the training corpus, vocabulary size, merge rules, and input text.

The useful property is that common byte sequences can become reusable tokens.

In [11]:
tests = [
    "In this BPE tokenizer",
    "token token token",
    "Unicode 😁",
    "support Unicode"
]

for sample in tests:
    byte_count = len(sample.encode("utf-8"))
    token_count = len(encode(sample))
    print(f"{sample!r}")
    print(f"  bytes  = {byte_count}")
    print(f"  tokens = {token_count}")
    print()

'In this BPE tokenizer'
  bytes  = 21
  tokens = 11

'token token token'
  bytes  = 17
  tokens = 6

'Unicode 😁'
  bytes  = 12
  tokens = 12

'support Unicode'
  bytes  = 15
  tokens = 14



## 12. A Small End-to-End Example

Suppose a text fragment starts as:

```text
[97, 98, 97, 98, 99]
```

and the learner discovers:

```text
(97, 98) → 256
```

The sequence becomes:

```text
[256, 256, 99]
```

Later, suppose:

```text
(256, 256) → 257
```

Then:

```text
[257, 99]
```

Token `257` therefore represents four underlying bytes:

```text
97 98 97 98
```

This recursive construction is the core idea behind BPE vocabulary growth.

In [12]:
toy_ids = [97, 98, 97, 98, 99]

step1 = merge(toy_ids, (97, 98), 256)
step2 = merge(step1, (256, 256), 257)

print("Original:", toy_ids)
print("After first merge: ", step1)
print("After second merge:", step2)

Original: [97, 98, 97, 98, 99]
After first merge:  [256, 256, 99]
After second merge: [257, 99]


# GPT-2-Style Tokenization

The next sections connect the simple BPE implementation to the tokenizer design used by GPT-2.

There are several distinct ideas here:

```text
Raw text
   ↓
GPT-2-style regex pre-tokenization
   ↓
Byte-level representation
   ↓
BPE merge rules
   ↓
Token IDs
```

The regex step and the BPE step are related, but they are not the same algorithm.

## 13. GPT-2-Style Regex Pre-Tokenization

GPT-2 uses a regex pattern to divide text into useful pieces before the byte-level BPE machinery is applied.

The pattern accounts for things such as:

- contractions such as `'s`, `'t`, `'re`, `'ve`, `'m`, `'ll`, `'d`
- letter sequences
- number sequences
- punctuation/symbol groups
- whitespace

The leading optional space in several alternatives is intentional: whitespace can become part of a tokenization unit.

In [13]:
import regex as re

gpt2_pattern = re.compile(
    r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"
)

samples = [
    "Hello World",
    "hello, world!",
    "I'm learning NLP",
    "GPT-2 has 1 tokenizer."
]

for sample in samples:
    print(sample)
    print(re.findall(gpt2_pattern, sample))
    print()

Hello World
['Hello', ' World']

hello, world!
['hello', ',', ' world', '!']

I'm learning NLP
['I', "'m", ' learning', ' NLP']

GPT-2 has 1 tokenizer.
['GPT', '-', '2', ' has', ' 1', ' tokenizer', '.']



### Why pre-tokenize at all?

Without this step, the BPE learner would consider every adjacent byte sequence across the whole raw string.

Pre-tokenization gives the tokenizer a structured starting point and changes which byte sequences are allowed to participate together. In GPT-2-style tokenization, this design is part of the tokenizer specification, not merely a cosmetic preprocessing step.

In [14]:
sample = "   hello world "
pieces = re.findall(gpt2_pattern, sample)

print("Input:", repr(sample))
print("GPT-2-style pieces:", pieces)

Input: '   hello world '
GPT-2-style pieces: ['  ', ' hello', ' world', ' ']


## 14. Compare with `tiktoken`

`tiktoken` is a fast tokenizer library. Here it is used as a reference implementation so we can inspect how GPT-2 token IDs are produced.

The from-scratch implementation above is intentionally small and educational; `tiktoken` is optimized for practical use.

In [15]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")

sample = "   hello world "
encoded_gpt2 = enc.encode(sample)

print("Text:", repr(sample))
print("Token IDs:", encoded_gpt2)
print("Decoded:", enc.decode(encoded_gpt2))

assert enc.decode(encoded_gpt2) == sample
print("tiktoken round-trip check passed.")

Text: '   hello world '
Token IDs: [220, 220, 23748, 995, 220]
Decoded:    hello world 
tiktoken round-trip check passed.


## 15. GPT-2 Vocabulary Files

A classic GPT-2 tokenizer distribution contains two central files.

### `encoder.json`

A JSON mapping from token strings to token IDs.

### `vocab.bpe`

The learned BPE merge rules, represented as pairs of symbols.

Together, these files define the vocabulary and merge ordering needed to reproduce GPT-2-style tokenization.

In [16]:
GPT2_BASE = "https://openaipublic.blob.core.windows.net/gpt-2/models/1558M"

encoder_path = Path("encoder.json")
bpe_path = Path("vocab.bpe")

if not encoder_path.exists():
    urlretrieve(f"{GPT2_BASE}/encoder.json", encoder_path)

if not bpe_path.exists():
    urlretrieve(f"{GPT2_BASE}/vocab.bpe", bpe_path)

print("Tokenizer files available:")
print("-", encoder_path)
print("-", bpe_path)

Tokenizer files available:
- encoder.json
- vocab.bpe


In [17]:
with open(encoder_path, "r", encoding="utf-8") as f:
    encoder = json.load(f)

with open(bpe_path, "r", encoding="utf-8") as f:
    bpe_data = f.read()

bpe_merges = [
    tuple(line.split())
    for line in bpe_data.splitlines()[1:]
    if line.strip()
]

print("Encoder entries:", len(encoder))
print("BPE merge rules:", len(bpe_merges))

special_token = "<|endoftext|>"
print(f"{special_token!r} ID:", encoder.get(special_token))

Encoder entries: 50257
BPE merge rules: 50000
'<|endoftext|>' ID: 50256


## 16. Special Tokens

Special tokens are different from ordinary learned text pieces.

For GPT-2, `<|endoftext|>` is a special vocabulary item used to mark a text boundary. It is represented by a reserved token ID in the tokenizer vocabulary.

The general lesson is that a tokenizer can contain both:

```text
ordinary tokens
special control/boundary tokens
```

These should not be confused when reasoning about the vocabulary.

In [18]:
special_token = "<|endoftext|>"

if special_token in encoder:
    token_id = encoder[special_token]
    print("Special token:", special_token)
    print("Token ID:", token_id)
else:
    print("Special token was not found in the loaded encoder.")

Special token: <|endoftext|>
Token ID: 50256


# SentencePiece

SentencePiece is a separate tokenizer framework that can train subword models directly from raw text.

It can be configured to train different model types, including BPE. Unlike a pipeline that first relies on conventional whitespace tokenization, SentencePiece is designed to learn segmentation directly from text while controlling normalization and splitting behavior through its configuration.

The following example intentionally uses a tiny toy corpus for learning. A real language-model tokenizer would be trained on a much larger and carefully prepared corpus.

In [19]:
import sentencepiece as spm

toy_text = (
    "SentencePiece is an unsupervised text tokenizer and detokenizer mainly for "
    "neural network based language models. "
    "It can learn subword units directly from raw text. "
) * 20

toy_path = Path("toy.txt")
toy_path.write_text(toy_text, encoding="utf-8")

print("Toy corpus written to:", toy_path)
print("Character count:", len(toy_text))

Toy corpus written to: toy.txt
Character count: 3280


## 17. Train a SentencePiece BPE Model

The configuration below keeps several of the ideas present in the original notebook:

- `model_type="bpe"` selects BPE.
- `vocab_size=400` gives a small vocabulary for the demo.
- `byte_fallback=True` provides a byte-level fallback path.
- `normalization_rule_name="identity"` disables additional normalization.
- `split_by_unicode_script`, `split_by_whitespace`, and `split_by_number` control segmentation behavior.
- `unk`, `bos`, and `eos` define special token IDs.

These settings are demonstration parameters, not a recommendation for training a production model.

In [20]:
sentencepiece_prefix = "tok400"

spm.SentencePieceTrainer.train(
    input=str(toy_path),
    input_format="text",
    model_prefix=sentencepiece_prefix,
    model_type="bpe",
    vocab_size=400,
    normalization_rule_name="identity",
    remove_extra_whitespaces=False,
    input_sentence_size=200000000,
    max_sentence_length=4192,
    seed_sentencepiece_size=1000000,
    shuffle_input_sentence=True,
    character_coverage=0.99995,
    byte_fallback=True,
    split_digits=True,
    split_by_unicode_script=True,
    split_by_whitespace=True,
    split_by_number=True,
    max_sentencepiece_length=16,
    add_dummy_prefix=True,
    allow_whitespace_only_pieces=True,
    unk_id=0,
    bos_id=1,
    eos_id=2,
    pad_id=-1,
    num_threads=os.cpu_count(),
)

print("SentencePiece model created.")
print(f"- {sentencepiece_prefix}.model")
print(f"- {sentencepiece_prefix}.vocab")

SentencePiece model created.
- tok400.model
- tok400.vocab


## 18. Encode and Decode with SentencePiece

After training, we can load the model and inspect both integer IDs and token pieces.

The round-trip assertion verifies:

```text
text → IDs → decoded text
```

returns the same string for this example.

In [21]:
sp = spm.SentencePieceProcessor(
    model_file=f"{sentencepiece_prefix}.model"
)

sample = "SentencePiece learns subword units."

ids = sp.encode(sample, out_type=int)
pieces = sp.encode(sample, out_type=str)
reconstructed = sp.decode(ids)

print("Input:        ", sample)
print("Token IDs:    ", ids)
print("Token pieces: ", pieces)
print("Decoded:      ", reconstructed)

assert reconstructed == sample
print("\nSentencePiece round-trip check passed.")

Input:         SentencePiece learns subword units.
Token IDs:     [360, 346, 381, 352, 347, 388]
Token pieces:  ['▁SentencePiece', '▁learn', 's', '▁subword', '▁units', '.']
Decoded:       SentencePiece learns subword units.

SentencePiece round-trip check passed.


# 19. Final Comparison

The three ideas explored here should be kept conceptually separate.

| Component | Main role |
|---|---|
| UTF-8 | Converts Unicode text into bytes |
| Byte-level BPE | Learns frequent byte-sequence merges |
| GPT-2 regex | Pre-tokenizes text before BPE |
| `tiktoken` | Practical tokenizer implementation |
| SentencePiece | Framework for training/using subword tokenizers |

A useful mental model is:

```text
Text
  │
  ├── UTF-8 bytes
  │      │
  │      └── BPE merges → learned token IDs
  │
  └── GPT-2-style regex
         │
         └── byte-level BPE → token IDs

SentencePiece is a different tokenizer framework that can itself
train a BPE model from raw text.
```
